# ch01 练习：Series 与 DataFrame 基础

> **怎么用**：打开本文件 → 找到所有 `# TODO(n)` → 把 `____` 替换成你认为正确的代码 →
> 直接 `Run All`。**跑通即为正确**，不需要翻答案。
> 全部卡住再看同目录 `ch01_series_dataframe_solution.ipynb`。

- 共 8 题，其中最后 1 题是跨小节综合题
- 难度：TODO(1)~(3) 单方法直用 → TODO(4)~(7) 易错场景 → TODO(8) 综合
- 建议限时：**25 分钟**（模拟竞赛上机节奏）

In [ ]:
import warnings

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

print("pandas", pd.__version__)

---

### TODO(1) 用字典构造 DataFrame

下面已经给好了 `data` 字典。请把它构造成 DataFrame，变量名 `df`，
**列顺序必须与字典的键顺序一致**。

In [ ]:
data = {
    "线路名称": ["LINE_ALPHA_1", "LINE_BETA_2", "LINE_GAMMA_3", "LINE_DELTA_4", "LINE_EPSILON_5"],
    "巡检时长": [1.5, 2.0, 0.75, 3.25, 1.0],
    "缺陷数": [2, 0, 5, 1, 3],
    "是否带电作业": [True, False, True, False, False],
}

# TODO(1)：把上面的字典构造成 DataFrame，变量名 df
# 提示：pandas.DataFrame(...)
df = ____

# ---- 验收 ----
assert isinstance(df, pd.DataFrame), "df 必须是一个 DataFrame"
assert df.shape == (5, 4), "形状不对，检查行数/列数"
assert list(df.columns) == ["线路名称", "巡检时长", "缺陷数", "是否带电作业"], "列顺序不齐"
assert df["缺陷数"].dtype.kind == "i", "缺陷数应该是整数类型"
print("TODO(1) 通过 ->\n", df)

### TODO(2) 带自定义索引的 Series + 标签取值

用给定的 `values` 和 `index` 构造一个 Series（变量名 `load`），
然后取出标签为 `STATION_C_01` 的那个值存入 `target`。

In [ ]:
values = [512.4, 96.2, 880.7, 45.1]
labels = ["STATION_A_01", "STATION_B_02", "STATION_C_01", "STATION_D_03"]

# TODO(2)：构造 Series（变量名 load），再按标签取出 STATION_C_01 的值存入 target
# 提示：pd.Series(data, index=...) ；取值用 Series.loc[标签] 或 at
load = ____
target = ____

# ---- 验收 ----
assert isinstance(load, pd.Series), "load 必须是 Series"
assert list(load.index) == labels, "索引不对"
assert load.name == "负荷值", "别忘了给 Series 起个 name"
assert abs(target - 880.7) < 1e-9, "取到的值不对，检查是不是用了 iloc 的位置语义"
print("TODO(2) 通过 -> target =", target)

### TODO(3) `loc` 取子表

用 `df` 取出：
- 行标签 **1 到 3（含两端）**
- 列 **「巡检时长」和「缺陷数」**

结果存入 `sub`。

In [ ]:
# TODO(3)：用 loc 取出 行标签 1~3（含）+ 两列，存入 sub
# 提示：DataFrame.loc[行切片, 列名列表] ；loc 的切片是左闭右闭
sub = ____

# ---- 验收 ----
assert isinstance(sub, pd.DataFrame), "sub 应该是 DataFrame"
assert sub.shape == (3, 2), f"形状应为 (3, 2)，实际 {sub.shape}"
assert list(sub.index) == [1, 2, 3], "行标签不对，注意 loc 切片含右端"
assert list(sub.columns) == ["巡检时长", "缺陷数"], "列不对"
print("TODO(3) 通过 ->\n", sub)

### TODO(4) `iloc` 取子表（左闭右开）

用 `iloc` 取出**前 4 行**的**第 1、2 列（位置 0 和 1）**，存入 `part`。

注意这题故意和上一题形成对比：同样看起来是「取一段」，`iloc` 的右端**不含**。

In [ ]:
# TODO(4)：用 iloc 取前 4 行、位置 0/1 两列，存入 part
# 提示：DataFrame.iloc[行位置切片, 列位置列表]
part = ____

# ---- 验收 ----
assert part.shape == (4, 2), f"形状应为 (4, 2)，实际 {part.shape}"
assert list(part.columns) == ["线路名称", "巡检时长"], "列不对，位置 0/1 就是前两列"
assert part.index.tolist() == [0, 1, 2, 3], "行不对"
print("TODO(4) 通过 ->\n", part)

### TODO(5) 布尔索引：多条件筛选

筛选出**「缺陷数 >= 2」且「巡检时长 < 3」**的记录，存入 `risk`。

提示：多条件要各自加括号，用 `&` 连接。

In [ ]:
# TODO(5)：多条件筛选，存入 risk
# 提示：(条件1) & (条件2) ；不能用 and
risk = ____

# ---- 验收 ----
assert isinstance(risk, pd.DataFrame), "risk 应该是 DataFrame"
assert set(risk["线路名称"]) == {"LINE_ALPHA_1", "LINE_GAMMA_3", "LINE_EPSILON_5"}, "筛出来的行不对"
assert risk.shape[0] == 3, "行数不对，检查是不是漏了某个条件"
print("TODO(5) 通过 ->\n", risk[["线路名称", "巡检时长", "缺陷数"]])

### TODO(6) 安全赋值（本章最大的坑）

在 `work` 这个副本上：
1. 新增一列 `风险等级`，默认全都是 `"低"`
2. 把**「缺陷数 >= 4」**的行的 `风险等级` 改成 `"高"`

**必须用 `loc` 的条件赋值写法**——用 `work[work["缺陷数"] >= 4]["风险等级"] = "高"`
在 pandas 3.0 下会静默失效。

In [ ]:
work = df.copy()
high_mask = work["缺陷数"] >= 4

# TODO(6)：先新增「风险等级」列、默认整列都是「低」；再把 high_mask 命中的行改成「高」
# 提示：新增列直接给列名赋值；条件赋值用 DataFrame.loc[布尔条件, 列名] = 新值
work["风险等级"] = ____
work.loc[high_mask, "风险等级"] = ____

# ---- 验收 ----
assert "风险等级" in work.columns, "没有新增列"
assert work["风险等级"].tolist() == ["低", "低", "高", "低", "低"], "赋值结果不对"
assert df.shape == (5, 4), "原表 df 被改动了，说明对副本的操作穿透回了原表"
print("TODO(6) 通过 ->\n", work[["线路名称", "缺陷数", "风险等级"]])

### TODO(7) 索引对齐

给定两个索引**不完全相同**的 Series，计算 `s1 + s2` 存入 `total`，
并统计 `total` 中有多少个缺失值存入 `n_missing`。

In [ ]:
s1 = pd.Series([10.0, 20.0, 30.0], index=["a", "b", "c"])
s2 = pd.Series([1.0, 2.0, 3.0], index=["b", "c", "d"])

# TODO(7)：计算 s1 + s2 存入 total；统计 total 的缺失个数存入 n_missing
# 提示：直接相加，pandas 会按标签对齐；缺失计数用 Series.isna().sum()
total = ____
n_missing = ____

# ---- 验收 ----
assert isinstance(total, pd.Series), "total 应该是 Series"
assert set(total.index) == {"a", "b", "c", "d"}, "索引应该是两者的并集"
assert abs(total.loc["b"] - 21.0) < 1e-9, "b 位置的值不对"
assert n_missing == 2, f"缺失个数应为 2，实际 {n_missing}"
print("TODO(7) 通过 ->", total.to_dict())

---

## 综合题：真实数据上的数据探索五步

下面用本章合成的设备缺陷数据 `data/device_defects.csv`（270 行 x 13 列）
走一遍竞赛里「数据探索」的标准动作。请按 4 个 `TODO` 依次完成。

| 步骤 | 产物 | 对应竞赛评分点 |
|---|---|---|
| ① 看结构 | `size` | 数据探索（形状/规模） |
| ② 选列 | `sub` | 数据探索（字段识别） |
| ③ 按标签取段 | `seg` | 数据清洗（定位目标记录） |
| ④ 派生新列 | `flagged` | 数据清洗（条件标记） |

In [ ]:
dev = pd.read_csv("data/device_defects.csv")
print("读入完成：", dev.shape)
print(dev.dtypes.to_string())

#### 综合题 ① 看结构

In [ ]:
# TODO(8)：把「行数, 列数」组成的元组存入 size
# 提示：DataFrame.shape 返回的就是 (行, 列)
size = ____

# ---- 验收 ----
assert size == (270, 13), f"应为 (270, 13)，实际 {size}"
assert len(dev.columns) == 13, "列数不对"
print("综合 ① 通过 -> size =", size)

#### 综合题 ② 选列（`loc` 的列选择形式）

In [ ]:
subset_cols = ["台区编号", "设备类型", "负荷值"]

# TODO(9)：取出 subset_cols 这三列存入 sub（保持原行数、原列顺序）
# 提示：DataFrame.loc[:, 列名列表] ；或直接用 df[列名列表]
sub = ____

# ---- 验收 ----
assert list(sub.columns) == subset_cols, "列的选取或顺序不对"
assert sub.shape == (dev.shape[0], 3), "行数应该保持和原表一致"
print("综合 ② 通过 -> sub.shape =", sub.shape)

#### 综合题 ③ 按标签取一段（`loc` 左闭右闭）

取**行标签 10 到 19（含两端）**、上面那三列，存入 `seg`。

In [ ]:
# TODO(10)：用 loc 取行标签 10~19（含）、subset_cols 三列，存入 seg
# 提示：dev 读进来是默认 RangeIndex，所以标签就是行号
seg = ____

# ---- 验收 ----
assert seg.shape == (10, 3), f"形状应为 (10, 3)，实际 {seg.shape}"
assert seg.index.tolist() == list(range(10, 20)), "行标签不对，别忘了 loc 切片含右端"
print("综合 ③ 通过 ->\n", seg.head(3))

#### 综合题 ④ 派生新列（条件标记）

在 `seg` 的**副本**上新增一列 `是否高负荷`：
- `负荷值 > 600` → `"是"`
- 其余 → `"否"`

结果存入 `flagged`（不要改动 `seg`）。

In [ ]:
high_load = seg["负荷值"] > 600

# TODO(11)：在 seg 副本上派生「是否高负荷」列：默认整列「否」，high_load 命中的行改「是」
# 提示：flagged = seg.copy()，再整列赋「否」，再 DataFrame.loc[条件, 列名] = 「是」
flagged = ____
flagged["是否高负荷"] = ____
flagged.loc[high_load, "是否高负荷"] = ____

# ---- 验收 ----
assert isinstance(flagged, pd.DataFrame), "flagged 应该是 DataFrame"
assert "是否高负荷" in flagged.columns, "没有新增列"
assert set(flagged["是否高负荷"]) <= {"是", "否"}, "取值只能是 是和 否"
expected_high = int((seg["负荷值"] > 600).sum())
assert (flagged["是否高负荷"] == "是").sum() == expected_high, "标记「是」的行数不对"
assert "是否高负荷" not in seg.columns, "seg 被改动了，应该在副本上操作"
print("综合 ④ 通过；高负荷记录数 =", expected_high)
print(flagged.head(5))

---

### 复盘提问

把没跑通的那题记进 `_notes/错题本.md`，写清「错在哪 + 正确写法」。